# Preprocesado y Feature Engineering
En este notebook realizaremos la agregación diaria de las ventas y generaremos variables auxiliares para capturar la estacionalidad. Finalmente, dividiremos los datos en los conjuntos de entrenamiento y test especificados en el planteamiento del problema.

In [2]:
import pandas as pd

# Se carga el dataset que se ha procesado anteriormente
df = pd.read_csv('../data/processed/data_cleaned.csv', parse_dates=['InvoiceDate'])

# Procedemos a extraer únicamente la fecha para poder agrupar
df['Date'] = df['InvoiceDate'].dt.date

# Procedemos a sumar el número de transacciones para las ventas por día
daily_sales = df.groupby('Date').agg({
    'Sales': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).reset_index()

daily_sales.rename(columns={'InvoiceNo': 'TransactionCount', 'CustomerID': 'UniqueCustomers'}, inplace=True)
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
daily_sales.head()

,Date,Sales,TransactionCount,UniqueCustomers
0,2010-12-01,36764.77,117,94
1,2010-12-02,38815.05,131,96
2,2010-12-03,23163.31,56,50
3,2010-12-05,30294.16,87,75
4,2010-12-06,26699.68,93,81


In [3]:
# Generamos variables temporales para capturar estacionalidad
daily_sales['DayOfWeek'] = daily_sales['Date'].dt.dayofweek
daily_sales['DayOfMonth'] = daily_sales['Date'].dt.day
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['IsWeekend'] = daily_sales['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Creamos "Lags", es decir, ventas del día anterior, para dar memoria al modelo
daily_sales['Sales_Lag1'] = daily_sales['Sales'].shift(1)
daily_sales = daily_sales.dropna()

In [4]:
from sklearn.preprocessing import StandardScaler

# Realizamos codificación One-Hot para el día de la semana
daily_sales = pd.get_dummies(daily_sales, columns=['DayOfWeek'], prefix='Day')

# Realizamos escalado de la variable objetivo
scaler = StandardScaler()

# Definimos fechas de corte según el planteamiento del problema
split_date = pd.to_datetime('2011-11-09')

# Dividimos el conjuntos de datos en datos de entrenamiento y datos de test
train_val_set = daily_sales[daily_sales['Date'] < split_date]
test_set = daily_sales[daily_sales['Date'] >= split_date]

print(f"[INFO] Días para Entrenamiento/Validación: {len(train_val_set)}")
print(f"[INFO] Días para Test (Evaluación Final): {len(test_set)}")

# Guardar los archivos finales para el modelado
train_val_set.to_csv('../data/processed/train_val.csv', index=False)
test_set.to_csv('../data/processed/test.csv', index=False)

[INFO] Días para Entrenamiento/Validación: 277
[INFO] Días para Test (Evaluación Final): 27
